# Week 3: CNN Classification and Kaggle Workflow

This notebook trains a small CNN on Fashion-MNIST, compares it with the Week 2 MLP, and contains the Kaggle Digit Recognizer workflow.

## 1. Imports and setup

In [ ]:
from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

PROJECT_ROOT = next((folder for folder in (Path.cwd(), *Path.cwd().parents)
                     if (folder / "week1" / "week1_ml_basics.ipynb").is_file()
                     and (folder / "week5" / "week5_vae.ipynb").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter in the research project folder or one of its subfolders.")
WEEK3_ROOT = PROJECT_ROOT / "week3"
DATA_DIR = PROJECT_ROOT / "data"

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)


## 2. Load Fashion-MNIST

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,)),
])

full_train_dataset = datasets.FashionMNIST(
    root=str(DATA_DIR), train=True, download=True, transform=transform
)
test_dataset = datasets.FashionMNIST(
    root=str(DATA_DIR), train=False, download=True, transform=transform
)

train_loader = DataLoader(full_train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128)

print(len(full_train_dataset), len(test_dataset))


## 3. Define the CNN

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = CNN().to(device)
print(model)


## 4. Define training and evaluation functions

In [ ]:
def run_epoch(model, loader, loss_fn, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)
    total_loss = 0.0
    correct = 0
    total = 0

    context = torch.enable_grad() if is_training else torch.inference_mode()
    with context:
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if is_training:
                optimizer.zero_grad()
            logits = model(images)
            loss = loss_fn(logits, labels)
            if is_training:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


## 5. Train the CNN

In [ ]:
epochs = 5
history = {"train_loss": [], "train_accuracy": [], "test_accuracy": []}

for epoch in range(epochs):
    train_loss, train_accuracy = run_epoch(model, train_loader, loss_fn, optimizer)
    _, test_accuracy = run_epoch(model, test_loader, loss_fn)
    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["test_accuracy"].append(test_accuracy)
    print(f"Epoch {epoch + 1}: loss={train_loss:.4f}, test accuracy={test_accuracy:.4f}")


## 6. Plot the CNN results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], marker="o")
axes[0].set(title="CNN training loss", xlabel="Epoch", ylabel="Loss")
axes[0].grid(True)
axes[1].plot(history["test_accuracy"], marker="o", label="Test")
axes[1].set(title="CNN accuracy", xlabel="Epoch", ylabel="Accuracy")
axes[1].legend()
axes[1].grid(True)
plt.tight_layout()
plt.show()


## 7. Compare with the Week 2 MLP

In [ ]:
# Replace the MLP value with the final value from your Week 2 run if you rerun it.
comparison = pd.DataFrame({
    "Model": ["MLP", "CNN"],
    "Test Accuracy": [0.8804, history["test_accuracy"][-1]],
})
comparison


## 8. Kaggle Digit Recognizer: load the CSV files

In [ ]:
# Download train.csv and test.csv from Kaggle and place them in this folder before running this section.
train = pd.read_csv(WEEK3_ROOT / "train.csv")
test = pd.read_csv(WEEK3_ROOT / "test.csv")

y = torch.tensor(train.pop("label").values).long()
X = torch.tensor(train.values / 255.0, dtype=torch.float32).view(-1, 1, 28, 28)
X_test = torch.tensor(test.values / 255.0, dtype=torch.float32).view(-1, 1, 28, 28)

print(X.shape, y.shape, X_test.shape)


## 9. Kaggle validation and submission

In [ ]:
# Use the same CNN class with one-channel 28x28 inputs.
num_train = int(0.8 * len(X))
indices = torch.randperm(len(X), generator=torch.Generator().manual_seed(seed))
train_idx, validation_idx = indices[:num_train], indices[num_train:]

kaggle_train_loader = DataLoader(TensorDataset(X[train_idx], y[train_idx]), batch_size=128, shuffle=True)
kaggle_validation_loader = DataLoader(TensorDataset(X[validation_idx], y[validation_idx]), batch_size=128)

# Train a fresh model with the training functions above.
kaggle_model = CNN().to(device)
kaggle_optimizer = torch.optim.Adam(kaggle_model.parameters(), lr=0.001)
for epoch in range(5):
    run_epoch(kaggle_model, kaggle_train_loader, loss_fn, kaggle_optimizer)
validation_loss, validation_accuracy = run_epoch(kaggle_model, kaggle_validation_loader, loss_fn)
print("Validation accuracy:", validation_accuracy)

test_logits = []
with torch.inference_mode():
    for start in range(0, len(X_test), 128):
        batch = X_test[start:start + 128].to(device)
        test_logits.append(kaggle_model(batch).argmax(1).cpu())
predictions = torch.cat(test_logits).numpy()
submission = pd.DataFrame({"ImageId": np.arange(1, len(predictions) + 1), "Label": predictions})
submission.to_csv(WEEK3_ROOT / "kaggle_submission_rerun.csv", index=False)
print(submission.head())


## Notes

The submitted `kaggle_submission.csv` received a Kaggle score of **0.98353**. A separate local rerun reached **0.98488** validation accuracy and wrote `kaggle_submission_rerun.csv`; this is not claimed to be the exact model behind the submitted score.
